In [ ]:
#!/usr/bin/env python
# coding: utf-8
"""
Telesca-Lovallo Visibility Graph Network — Italy INGV Catalog (1985-2025)

Run as a script  : python ITALY_network_TL.py
Convert to notebook: python convert_to_notebook.py ITALY_network_TL.py notebooks/ITALY_network_TL.ipynb
"""

import logging
import pickle
import time
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
import seaborn as sns

from src.network import build_telesca_lovallo_network
from src.metrics import estimate_gamma_mle, test_power_law, measure_pa_growing_graph, plot_preferential_attachment
from src.assortativity import compute_assortativity, plot_assortativity, plot_knn, plot_rich_club
from src.centrality import (
    plot_centrality_correlation,
    plot_top_n_cells,
    plot_geo_top_n_interactive,
    plot_geo_centrality_overlap,
    compute_bb_fitness_events,
    plot_bb_fitness,
    plot_bb_fitness_theory,
    plot_bb_fitness_geo,
)
from src.community import (
    run_louvain,
    run_consensus_louvain,
    run_spectral,
    run_infomap,
    run_directed_louvain,
    run_hdbscan_spectral,
    run_hdbscan_geo,
    run_bigclam,
    compare_partitions,
    plot_partition_scores,
    compute_nmi_matrix,
    plot_nmi_heatmap,
    plot_community_geo,
)
from src.viz import (
    analyze_degree_distribution,
    analyze_degree_distribution_log_binning,
    plot_ccdf_with_fit,
    plot_degree_distribution_linear,
)
from src.plotutils import setup_matplotlib, configure_saves, savefig, save_plotly

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[assignment]

logging.basicConfig(level=logging.INFO, format="%(levelname)s  %(message)s")
sns.set_theme(style="whitegrid")
pio.renderers.default = "notebook"

DATA_DIR    = Path("data/INGV")
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)
(RESULTS_DIR / "data").mkdir(exist_ok=True)
(RESULTS_DIR / "cache").mkdir(exist_ok=True)
(RESULTS_DIR / "gephi").mkdir(exist_ok=True)

CUT_YEAR = 1985

# TL model parameter
TL_MAX_LOOK_AHEAD = 10_000   # safety cap; natural slope termination is faster in practice

SAVE_PDF: bool = True
SAVE_JPG: bool = True

setup_matplotlib()
configure_saves(SAVE_JPG, SAVE_PDF, RESULTS_DIR / "figures" / "italy" / "tl")

## Data Loading

The INGV catalog covers Italian seismicity 1985–2025 at M ≥ 1.5.  Events are
filtered to `year ≥ 1985` (catalog completeness threshold) and sorted by UTC
origin time.  The integer reset index is the node ID used in the TL and HVG
analyses, where nodes are individual events in time-series order.

The TL / NVG model treats the earthquake catalog as a **magnitude time series**:
only the sequence $(m_1, m_2, \ldots, m_N)$ and the corresponding times $(t_i)$
are used; spatial coordinates (latitude, longitude) enter only as node attributes
for geographic visualisation.

**Expected output:** ≈ 215,000 rows, RAM ≈ 30–60 MB.  The head/tail table
confirms chronological ordering and that all required columns are present.

In [ ]:
print("Loading Italy earthquake catalog...")
df = pd.read_csv(DATA_DIR / "italy_earthquakes_1985_2025.csv")
df["time"] = pd.to_datetime(df["time"], utc=True)
df_net = (
    df[df["time"].dt.year >= CUT_YEAR]
    .sort_values("time")
    .reset_index(drop=True)
)

print(f"Loaded {len(df_net):,} earthquakes "
      f"({df_net['time'].dt.year.min()}–{df_net['time'].dt.year.max()})")
print(f"RAM: {df_net.memory_usage(deep=True).sum() / 1024**2:.1f} MB")
display(pd.concat([df_net.head(3), df_net.tail(3)]))

## Network Construction

The **Natural Visibility Graph** (NVG) maps a time series to a network by
connecting any two observations that can "see" each other: event $i$ and
event $j$ ($i < j$ in time) are linked when the straight line joining their
$(t, m)$ coordinates lies strictly above all intermediate observations:

$$m_k < m_i + (m_j - m_i)\frac{t_k - t_i}{t_j - t_i}, \quad \forall\, i < k < j$$

Using the angular-slope identity this simplifies to:

$$s_{ij} = \frac{m_j - m_i}{t_j - t_i} > \max_{i < k < j} s_{ik}$$

which admits an $O(n)$ right-side scan per event with a running maximum slope —
no nested loop over intermediates is needed.  Every adjacent pair $(i, i+1)$ is
trivially connected (no intermediates), so the NVG is **always connected**:
it contains the path $0 - 1 - \cdots - N-1$ as a subgraph.

Unlike the ABE (cell-level) and BP (parent–child) models, the NVG is a pure
**time-series construction**: it encodes the local and global amplitude structure
of the magnitude sequence without any spatial information.  High-degree nodes
are magnitude-dominant events visible over long stretches of the series —
the topological signature of mainshocks in the visibility graph.

*References:* Luque et al. (2009) *Phys. Rev. E*, 80, 046103;
Telesca & Lovallo (2012) *EPL*, 97, 50002.

**Expected output:** ≈ 215,000 nodes and ≈ 400,000–700,000 edges
($\langle k \rangle \approx 4$–$6$ for seismic NVG).  Build time ≈ 3–8 minutes
with the vectorised `np.maximum.accumulate` implementation.  Min degree = 2
(endpoint events that connect only to their neighbour).

In [ ]:
print(f"Building TL visibility graph (max_look_ahead={TL_MAX_LOOK_AHEAD})…")
t_build = time.time()
G = build_telesca_lovallo_network(df_net, max_look_ahead=TL_MAX_LOOK_AHEAD)
print(f"Build time: {time.time() - t_build:.1f}s")
print(f"Nodes: {G.number_of_nodes():,}  Edges: {G.number_of_edges():,}")
print(f"Average degree: {2*G.number_of_edges()/G.number_of_nodes():.2f}")
print(f"Min degree: {min(d for _, d in G.degree())}  "
      f"Max degree: {max(d for _, d in G.degree()):,}")

## Hub Map — 2D

High-degree events in the NVG are earthquakes that are simultaneously visible
over long stretches of the magnitude time series — amplitude-dominant events on
both their left and right.  These are large mainshocks whose magnitude dwarfs
neighbouring events, creating an unobstructed line-of-sight to many events.

Unlike ABE hubs (most-visited spatial cells) and BP hubs (most-productive
triggers), TL hubs are **purely magnitude-driven**: a M7 mainshock visible
across hundreds of subsequent aftershocks appears as a hub regardless of its
spatial location or depth.

Marker size is linearly scaled to [8, 35] pixels; points are rendered in
ascending degree order so the highest values appear on top.

**Expected output:** hubs are sparser than in ABE (where many cells tie for
high degree) and more spatially concentrated than BP (where the first event
in the catalog dominates PageRank).  Expect the largest documented mainshocks
to appear as prominent dots: L'Aquila 2009 (Mw 6.1), Amatrice 2016 (Mw 6.2),
Irpinia 1980 (Mw 6.9), and the 1990 Santa Lucia earthquake (Mw 5.6, Sicily).
Events from Greece / Ionian Islands are present in the catalog (INGV east
boundary = 22° E) and may appear at the map edge — they are valid catalog
entries, not errors.

In [ ]:
df_nodes = pd.DataFrame([
    {
        "event_idx": n,
        "degree":    G.degree(n),
        "lat":       G.nodes[n]["lat"],
        "lon":       G.nodes[n]["lon"],
        "magnitude": G.nodes[n]["magnitude"],
    }
    for n in G.nodes()
])
df_hubs = df_nodes.nlargest(100, "degree").copy()
print(f"Top 100 hubs: degree range [{df_hubs['degree'].min():.0f}, {df_hubs['degree'].max():.0f}]")

# Sort ascending so high-degree markers render on top in plotly
df_hubs = df_hubs.sort_values("degree")

# Scale size to [8, 35] range
_deg_min = df_hubs["degree"].min()
_deg_max = df_hubs["degree"].max()
df_hubs["marker_size"] = 8 + 27 * (df_hubs["degree"] - _deg_min) / max(_deg_max - _deg_min, 1)

_IT_BOUNDS = dict(west=3, east=22, south=34, north=48)
MAP_WIDTH  = 770
MAP_HEIGHT = 700

fig = px.scatter_map(
    df_hubs, lat="lat", lon="lon",
    color="degree", size="marker_size",
    size_max=40,
    color_continuous_scale="inferno",
    map_style="carto-positron",
    hover_name="event_idx",
    hover_data={"magnitude": ":.2f", "degree": True, "marker_size": False},
    title="TL Network Hubs: Top 100 Highest-Degree Events — Italy",
)
fig.update_layout(
    margin={"r": 0, "t": 40, "l": 0, "b": 0},
    width=MAP_WIDTH, height=MAP_HEIGHT,
    map=dict(center={"lat": 41.9, "lon": 12.5}, zoom=0, bounds=_IT_BOUNDS),
)
save_plotly(fig, "tl_hub_map_2d_italy")
fig.show()

## Degree Distribution — Linear

Linear-scale bar chart of node degrees, truncated at $k = 50$ and plotted on
a log y-axis.

The NVG minimum degree is 2 (endpoint events that connect only to their
immediate neighbour), so the distribution starts at $k = 2$ — unlike ABE and
BP where $k = 0$ events exist.  The expected distribution for an i.i.d. random
series is exponential; deviations toward a heavier tail indicate long-range
correlations in the magnitude sequence (e.g., aftershock clustering where a
large mainshock is visible across hundreds of low-magnitude aftershocks).

**Expected output:** a steeply declining bar chart starting at $k = 2$, with
a mode around $k = 3$–$5$ and a visible exponential-to-power-law tail.
Compared to BP, the TL distribution has a higher bulk degree (no $k = 0$ events)
but a lighter tail (the NVG condition is less exclusive than parent selection).

In [ ]:
plot_degree_distribution_linear(G, "TL Italy", max_degree=50, weighted=False)

## Degree Distribution — Log-Log

Log-log scatter of $P(k)$ vs $k$.  For an uncorrelated (i.i.d.) time series
the NVG degree distribution is exponential — a straight line on a log-linear
plot, not log-log.  A power-law tail on a log-log plot signals long-range
correlations or heavy-tailed amplitude fluctuations in the magnitude sequence.

Telesca & Lovallo (2012) reported power-law-like tails for seismic series,
attributing them to the clustering of aftershocks around large mainshocks:
a high-magnitude mainshock surrounded by many smaller aftershocks acquires a
very high degree because it is visible "over" all of them.

**Expected output:** a scatter that appears roughly linear on the log-log plot
for $k \geq 4$–$10$, transitioning to exponential-like decline for small $k$.
Compare the slope to the log-binned version and the MLE estimate below.

In [ ]:
analyze_degree_distribution(G, "TL Italy")

## Degree Distribution — Log Binning

Logarithmic binning (Clauset et al. 2009) reduces noise in the sparse high-$k$
tail by pooling observations into geometrically spaced bins and normalising by
bin width to yield probability *density* $P(k)$.  This is the standard approach
for heavy-tailed distributions where linear binning leaves the tail noisy.

**Expected output:** a cleaner curve than the raw log-log scatter.  The slope
of the fitted line for $k \geq 4$ gives a visual estimate of the tail exponent
$\gamma$ before the MLE step.  If the fit quality R² < 0.7, the distribution
may be better described by a log-normal or exponential rather than a power law.

In [ ]:
analyze_degree_distribution_log_binning(G, "TL Italy", k_min_fit=4)

## Degree Distribution — CCDF

The CCDF $P(K \geq k)$ is monotone non-increasing and avoids binning artefacts.
A straight line on a log-log CCDF confirms a power-law tail with slope
$-(\gamma - 1)$; an exponential CCDF appears linear on a log-linear plot.

**Expected output:** approximately linear CCDF for $k \geq 4$ with slope
≈ −1.5 to −2.5 (corresponding to $\gamma \approx 2.5$–$3.5$).  A concave
CCDF on the log-log plot is consistent with an exponential or sub-power-law
distribution — common in the bulk but rare in the tail of seismic NVGs.

In [ ]:
plot_ccdf_with_fit(G, "TL Italy", k_min_fit=4)

## MLE Gamma

Maximum-likelihood estimate of the tail exponent (Clauset et al. 2009):

$$\hat{\gamma} = 1 + n\left[\sum_{i=1}^{n}\ln\frac{k_i}{k_{\min}}\right]^{-1}$$

where the sum runs over $n$ events with $k \geq k_{\min} = 4$.

The CSN likelihood-ratio test compares the power-law to an exponential
alternative.  For a purely random i.i.d. series, the NVG degree distribution is
geometric and the exponential model wins ($R < 0$).  For a seismic series with
aftershock clustering, a positive $R$ with $p < 0.05$ indicates that magnitude
fluctuations produce a heavier-than-geometric visibility tail — evidence of
long-range dependence in the magnitude sequence.

*References:* Luque et al. (2009) proved analytically that i.i.d. uniform
series give $P(k) = (1/3)(2/3)^{k-2}$ (geometric, exponential tail).
Telesca & Lovallo (2012) found $\gamma \approx 2$–$4$ for seismic NVGs.

**Expected output:** $\hat{\gamma} \approx 2.5$–$4.0$; $\sigma_{MLE} \approx
0.1$–$0.3$.  CSN $R > 0$, $p < 0.05$ — power-law preferred over exponential,
confirming aftershock-driven long-range correlations in the magnitude time series.

In [ ]:
degs     = [d for _, d in G.degree()]
gamma_tl = estimate_gamma_mle(degs, k_min=4)
print(f"MLE γ (degree, k_min=4): {gamma_tl:.3f}")

result_tl = test_power_law(degs, k_min=4)
print(f"  CSN test: γ={result_tl['gamma']} (σ={result_tl['sigma']})  "
      f"R={result_tl['R']}  p={result_tl['p_value']}  → {result_tl['verdict']}")

## Preferential Attachment

The empirical attachment kernel $\pi(k) \propto k^{\alpha}$ measures whether
events with more visibility connections (high degree) attract further connections
at a rate proportional to their degree (Jeong, Néda & Barabási 2003).

$$\pi(k) = \frac{\sum_{i:\,k_i(t)=k} \Delta k_i}{\#\{i:\,k_i(t)=k\}}$$

In the NVG, each new event $j$ is added to the series and connects to all
earlier events $i < j$ that remain visible (not obscured by taller intermediate
events).  At the moment $j$ arrives, the degree of each earlier neighbour $i$
is recorded before $j$'s edges are added — the *batch arrival* convention
standard for growing graph PA estimation.

**Seismological interpretation:** $\alpha \approx 1$ would indicate that already
visible events (mainshocks that can "see" many neighbours) attract additional
visibility connections at a proportional rate — consistent with large events
acting as long-range anchors in the temporal magnitude landscape.
$\alpha > 1$ (super-linear) could indicate that the tallest mainshocks
increasingly dominate the NVG as the catalog grows.

*Reference:* Jeong H., Néda Z. & Barabási A.-L. (2003). Measuring preferential
attachment in evolving networks. *Europhysics Letters*, 61(4), 567–572.

In [ ]:
print("Measuring preferential attachment kernel (NVG growing graph)…")
ks_pa, pi_k_pa, alpha_pa = measure_pa_growing_graph(G, df_net)
plot_preferential_attachment(ks_pa, pi_k_pa, alpha_pa, title="TL Italy (NVG PA)")

## Macroscopic Metrics

The NVG is guaranteed connected (every adjacent pair is visible), so no giant
component extraction is needed.  Three metrics characterise the global topology:

1. **Average degree** $\langle k \rangle = 2M/N$.  For i.i.d. uniform series,
the NVG gives $\langle k \rangle = 4$.  Seismic catalogs typically show
$\langle k \rangle \approx 4$–$6$ (aftershock clustering extends visibility).

2. **Average clustering** $C$: fraction of event triples $(i,j,k)$ where all
three can mutually see each other.  High $C$ indicates aftershock bursts —
a cluster of low-magnitude events surrounding a large mainshock form a
near-clique because the mainshock is visible to all of them.
For a random i.i.d. series, $C_{\text{ER}} \approx \langle k \rangle / N \approx 0$.

3. **Approximate average path length** $L$ (500 BFS seeds): the mean hop
distance between random event pairs.  Exact computation is $O(N^2)$ for
215k nodes; the 500-seed approximation gives ±5% accuracy.

The **small-world signature** requires $C \gg C_{\text{ER}}$ with
$L \approx L_{\text{ER}} = \ln N / \ln \langle k \rangle$.  For NVG on seismic
series, $C / C_{\text{ER}} \approx 100$–$1000\times$ while $L / L_{\text{ER}}
\approx 1.5$–$3\times$ — confirming small-world structure driven by
aftershock cliques.

**Expected output:** $\langle k \rangle \approx 4$–$6$; $C \approx 0.2$–$0.4$;
$C/C_{\text{ER}} \approx 100$–$500\times$; $L \approx 15$–$35$ hops.

In [ ]:
N_nodes = G.number_of_nodes()
M_edges = G.number_of_edges()
avg_k   = 2 * M_edges / N_nodes
print(f"Nodes: {N_nodes:,}  Edges: {M_edges:,}  ⟨k⟩ = {avg_k:.2f}")

print("Computing clustering coefficient…")
avg_c = nx.average_clustering(G)
c_er  = avg_k / N_nodes
print(f"Avg clustering C = {avg_c:.4f}  (ER expected: {c_er:.4f}  ratio: {avg_c/max(c_er,1e-12):.1f}×)")

print("Approximating average path length (500 random seeds)…")
rng       = np.random.default_rng(42)
seeds_apl = list(rng.choice(N_nodes, size=min(500, N_nodes), replace=False))
apl_vals  = []
for s in seeds_apl:
    lengths = nx.single_source_shortest_path_length(G, s)
    apl_vals.extend(lengths.values())
avg_L = np.mean(apl_vals)
l_er  = np.log(N_nodes) / np.log(avg_k) if avg_k > 1 else float("nan")
print(f"Approx avg path length L ≈ {avg_L:.2f}  (ER expected: {l_er:.2f})")
print(f"Small-world: C/C_ER = {avg_c/max(c_er,1e-12):.1f}×,  L/L_ER = {avg_L/max(l_er,1e-12):.2f}×")

## Centrality Analysis

Nine centrality measures identify structurally important events in the undirected NVG:

- **Degree**: direct visibility count — amplitude-dominant events visible to
many others in both directions.
- **PageRank**: importance propagated through the visibility chain — events
near many high-degree nodes receive high PR even if their own degree is moderate.
- **Betweenness** ($k = 500$ sample): fraction of shortest paths passing through
each node.  On the NVG, events that separate distinct seismic episodes (between
two large mainshocks in the time series) have high betweenness as "bridge events".
- **Eigenvector / Katz**: neighbour-weighted importance — converges on the
same hub events as degree for tree-like graphs.
- **HITS Hub / Authority**: on the undirected NVG, Hub ≈ Authority ≈ Degree
(all measure amplitude dominance).
- **Harmonic centrality**: $C_H(v) = \sum_{u \neq v} 1/d(u,v)$, summing inverse
shortest-path distances. Handles disconnected components gracefully (zero
contribution from unreachable pairs). On the NVG, which can fragment during
seismically quiet intervals, harmonic centrality identifies events with the
broadest temporal reach — visible to many others across the full magnitude time
series even when the graph is not connected (*Bavelas 1950; Rochat 2009*).
- **Clustering coefficient**: fraction of a node's neighbour pairs that are
mutually connected ($C_i = 2t_i / k_i(k_i-1)$). High clustering on the NVG
indicates dense aftershock bursts where many consecutive events are mutually
visible — a signature of tight temporal clustering around a mainshock
(*Watts & Strogatz 1998*).
- **Triangle count**: raw number of closed triangles per node. Non-zero values
confirm the NVG has richer topology than a pure path chain; high counts
co-locate with the denser aftershock sequences.

Centralities are computed inline because `compute_all_centralities` requires
ABE-style "cx_cy_cz" cell-string node IDs (not applicable to integer event IDs).

**Expected output:** top-5 events across most metrics overlap heavily with the
hub-map top-100 (the same large mainshocks dominate all amplitude-based metrics).
Betweenness may diverge — it captures "bridge" events separating temporally
distinct sequences, which may not be the largest in magnitude.  Total runtime
≈ 10–20 minutes (dominated by betweenness with $k = 500$ samples).

In [ ]:
print("Computing centralities for TL network…")
_t0_cent = time.time()

G_nsl = G.copy()
G_nsl.remove_edges_from(nx.selfloop_edges(G_nsl))
_N    = G.number_of_nodes()

deg_cent = nx.degree_centrality(G)
print(f"  Degree done ({time.time()-_t0_cent:.1f}s)")

# PageRank on undirected graph (treat as symmetric DiGraph internally)
pr_cent = nx.pagerank(G, weight="weight")
print(f"  PageRank done ({time.time()-_t0_cent:.1f}s)")

bet_cent = nx.betweenness_centrality(G, k=min(500, _N), seed=42, normalized=True)
print(f"  Betweenness done ({time.time()-_t0_cent:.1f}s)")

try:
    eig_cent = nx.eigenvector_centrality(G, weight="weight", max_iter=500, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    eig_cent = nx.eigenvector_centrality_numpy(G, weight="weight")
print(f"  Eigenvector done ({time.time()-_t0_cent:.1f}s)")

_max_deg    = max(d for _, d in G.degree())
_alpha_katz = 0.85 / _max_deg
try:
    katz_cent = nx.katz_centrality(
        G, alpha=_alpha_katz, weight="weight", normalized=True, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    katz_cent = nx.katz_centrality_numpy(G, alpha=_alpha_katz, weight="weight")
print(f"  Katz done ({time.time()-_t0_cent:.1f}s)")

try:
    hits_hub, hits_auth = nx.hits(G_nsl, max_iter=1000, tol=1e-6)
except nx.PowerIterationFailedConvergence:
    _zeros    = {n: 0.0 for n in G.nodes()}
    hits_hub  = _zeros.copy()
    hits_auth = _zeros.copy()
print(f"  HITS done ({time.time()-_t0_cent:.1f}s total)")

harm_cent = nx.harmonic_centrality(G)
print(f"  Harmonic done ({time.time()-_t0_cent:.1f}s)")

clust_cent = nx.clustering(G, weight="weight")
print(f"  Clustering done ({time.time()-_t0_cent:.1f}s)")
tri_count = nx.triangles(G)
print(f"  Triangles done ({time.time()-_t0_cent:.1f}s)")

df_centrality = pd.DataFrame([
    {
        "cell_id":     n,
        "lat":         G.nodes[n]["lat"],
        "lon":         G.nodes[n]["lon"],
        "depth_km":    G.nodes[n]["depth_km"],
        "Degree":      deg_cent.get(n, 0.0),
        "PageRank":    pr_cent.get(n, 0.0),
        "Betweenness": bet_cent.get(n, 0.0),
        "Eigenvector": eig_cent.get(n, 0.0),
        "Katz":        katz_cent.get(n, 0.0),
        "HITS_Hub":    hits_hub.get(n, 0.0),
        "HITS_Auth":   hits_auth.get(n, 0.0),
        "Harmonic":    harm_cent.get(n, 0.0),
        "Clustering":  clust_cent.get(n, 0.0),
        "Triangles":   float(tri_count.get(n, 0)),
    }
    for n in G.nodes()
    if "lat" in G.nodes[n] and "lon" in G.nodes[n]
])

for metric, label in [
    ("Degree",      "Amplitude-Dominant Events"),
    ("PageRank",    "Propagation Influencers"),
    ("Betweenness", "Episode Bridges"),
    ("HITS_Hub",    "Visibility Hubs"),
    ("HITS_Auth",   "Visibility Authorities"),
    ("Harmonic",    "Topological Reach (Harmonic)"),
    ("Clustering",  "Triangle Density (Clustering)"),
    ("Triangles",   "Feedback Loops (Triangles)"),
]:
    print(f"\n--- Top 5: {label} ({metric}) ---")
    display(df_centrality.nlargest(5, metric)[
        ["cell_id", "lat", "lon", "depth_km", metric]])

plot_centrality_correlation(df_centrality, "TL Italy")
plot_top_n_cells(df_centrality, top_n=10, title="TL Italy")

## Centrality Geo Maps

Geographic projection of the top-10 events per centrality metric.  Two views:

1. **Per-metric map** — 7 panels, top-10 per centrality.
2. **Overlap map** — events coloured by how many metrics they top simultaneously.

Since TL hubs are magnitude-driven (not spatially driven), there is no prior
expectation that they cluster along fault zones — they should coincide with the
*largest historical mainshocks in the time series* regardless of location.

High convergence (warm colours in the overlap map) identifies events that are
simultaneously amplitude-dominant, well-connected in the visibility chain, and on
the shortest paths between many pairs — a structurally multi-dominant mainshock
signature.  These are the events a seismologist would classify as "turning points"
in the 40-year seismic record.

**Expected output:** L'Aquila 2009 and Amatrice 2016 should appear in nearly
every metric's top-10 given their magnitude and the dense aftershock sequences
they generated.  Betweenness may highlight additional events separating the
pre-2000 and post-2000 seismic episodes.

In [ ]:
plot_geo_top_n_interactive(
    df_centrality, top_n=10,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)
plot_geo_centrality_overlap(
    df_centrality, top_n=10,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
)

## Bianconi-Barabasi Fitness

The fitness estimator $\hat{\beta}_i = \ln k_i(T) / \ln(T/t_i)$ applied to
the NVG asks: among all events in the seismic catalog, which ones accumulated
visibility connections at an intrinsically rapid rate, controlling for how
early they arrived in the time series?

$$k_i(t) \approx m\!\left(\frac{t}{t_i}\right)^{\beta_i},\qquad
\hat{\beta}_i = \frac{\ln k_i(T)}{\ln(T/t_i)}$$

In the NVG context, $t_i$ is the earthquake occurrence time and $k_i(T)$
is the final number of visibility connections event $i$ holds.  High
$\hat{\beta}$ identifies events that remained "visible" from many subsequent
events throughout the catalog — structurally, these are the magnitude
peaks that dominate the visibility landscape.

Three theoretical regimes are compared (equal-fitness BA, uniform fitness,
Bose-Einstein condensation) against the observed $\rho(\hat{\beta})$.

*References:* Bianconi G. & Barabási A.-L. (2001). *EPL* 54, 436–442;
*PRL* 86, 5632–5635.

In [ ]:
print("Computing Bianconi-Barabasi fitness (NVG events)…")
df_bb = compute_bb_fitness_events(G, df_net)
print(f"  {len(df_bb)} events with valid β̂; β range [{df_bb['fitness_beta'].min():.3f}, {df_bb['fitness_beta'].max():.3f}]")
plot_bb_fitness(df_bb, title="TL Italy")
plot_bb_fitness_theory(df_bb, gamma=gamma_tl, title="TL Italy")
plot_bb_fitness_geo(
    df_bb,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0,
    bounds=_IT_BOUNDS, height=MAP_HEIGHT, width=MAP_WIDTH,
)

## Community Detection — Full Suite

Seven community-detection algorithms are applied to the undirected TL (NVG) graph.
Modularity

$$Q = \frac{1}{2m}\sum_{ij}\left[A_{ij} - \frac{k_i k_j}{2m}\right]\delta(c_i,c_j)$$

(Newman & Girvan 2004) is the primary optimisation target for graph-based methods,
but density- and affiliation-based methods operate on complementary representations.

In contrast to BP (communities = aftershock subtrees defined by spatial proximity),
TL communities are determined by the **amplitude structure of the magnitude
sequence**: a community is a set of events mutually visible to each other in the
NVG and separated by amplitude barriers from the rest of the catalog.
Seismologically, a community boundary corresponds to a large mainshock that creates
an amplitude barrier — events on opposite sides cannot see each other in the NVG
because the mainshock occludes their line-of-sight.  Communities therefore group
events within the same seismic episode bounded by two major mainshocks.

**Graph-based methods**
- **Louvain** (Blondel et al. 2008): greedy modularity maximisation; fast and
scalable but subject to resolution limit and stochastic variation.
- **Consensus Louvain** (Lancichinetti & Fortunato 2012): 100-run ensemble average
over co-assignment frequencies; the most reproducible partition.
- **Spectral** (Von Luxburg 2007): Fiedler-vector bipartition applied recursively;
run on the full graph since the NVG is always connected.
- **InfoMap** (undirected; Rosvall & Bergstrom 2008): minimises the description
length of a random walk; often finds finer-grained communities than modularity.

**Density-based methods**
- **HDBSCAN-Spectral** (Campello et al. 2013): applies HDBSCAN to the
$k$-dimensional spectral embedding of the normalised graph Laplacian.
The number of communities is determined entirely by the data — no $k$
pre-specification is required.  Points in low-density embedding regions are
labelled as noise and excluded from communities.
- **HDBSCAN-Geographic**: same HDBSCAN algorithm applied directly to projected
$(x, y)$ node coordinates in kilometres (EPSG:32632); no graph structure is used.
Comparing with HDBSCAN-Spectral tests whether the amplitude-barrier communities
detected by the NVG merely reflect geographic clustering or carry additional
temporal/seismic-episode information beyond spatial proximity.

**Overlapping-community method**
- **BigCLAM** (Yang & Leskovec 2013): solves for an $N \times k$ affiliation matrix
$F$ via non-negative matrix factorisation; the hard partition is recovered by
$\arg\max_c F_{ic}$.  Events near major mainshocks (amplitude barriers) may carry
affiliation in two adjacent communities, capturing the transition between seismic
episodes.

**Partition quality scoring**
All seven partitions are evaluated on 9 quality metrics — modularity, conductance,
coverage, Ncut, map equation, DC-SBM log-likelihood, Surprise, geographic
compactness, and depth coherence — via `compare_partitions`.

High NMI across graph-based methods confirms that amplitude-barrier communities are
robust and not algorithmic artefacts.

*References:* Newman & Girvan (2004) *Phys. Rev. E* 69, 026113;
Blondel et al. (2008) *J. Stat. Mech.* P10008;
Rosvall & Bergstrom (2008) *PNAS* 105, 1118;
Campello et al. (2013) *ECML-PKDD* 160–175;
Yang & Leskovec (2013) *ACM TKDD* 7(3), 1–42.

**Expected output:** Louvain finds 5–15 communities corresponding to major seismic
episodes.  NMI between Louvain and Consensus ≈ 0.4–0.7 (higher than BP because the
NVG has clearer amplitude barriers than the BP spanning tree).  Community geo maps
should show geographic coherence, with each community grouping events within the
same seismic episode bounded by two large mainshocks.

In [ ]:
print("Running community detection on TL graph…")

print("Louvain…")
community_map = run_louvain(G, seed=42)
k_louvain     = len(set(community_map.values()))
print(f"  {k_louvain} communities")
plot_community_geo(
    G, community_map,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Louvain",
)

print("Consensus Louvain (100 runs)…")
community_map_consensus = run_consensus_louvain(G, n_runs=100, seed=42)
print(f"  {len(set(community_map_consensus.values()))} communities")
plot_community_geo(
    G, community_map_consensus,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Consensus Louvain",
)

print(f"Spectral (k={min(k_louvain, 8)})…")
_k_spec = min(k_louvain, 8)
community_map_spectral = run_spectral(G, k=_k_spec, seed=42)
print(f"  {len(set(community_map_spectral.values()))} communities")
plot_community_geo(
    G, community_map_spectral,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Spectral",
)

print("InfoMap (undirected)…")
community_map_infomap = run_infomap(G, directed=False, seed=42)
print(f"  {len(set(community_map_infomap.values()))} communities")
plot_community_geo(
    G, community_map_infomap,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="InfoMap",
)

partitions = {
    "Louvain":   community_map,
    "Consensus": community_map_consensus,
    "Spectral":  community_map_spectral,
    "InfoMap":   community_map_infomap,
}
print("Computing NMI…")
nmi_df = compute_nmi_matrix(partitions)
display(nmi_df.round(3))
plot_nmi_heatmap(nmi_df, title="TL Italy")

print("HDBSCAN-Spectral…")
community_map_hdbscan_spec = run_hdbscan_spectral(G, min_cluster_size=10, seed=42)
print(f"  {len(set(community_map_hdbscan_spec.values()))} clusters")
plot_community_geo(
    G, community_map_hdbscan_spec,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Spectral",
)

print("HDBSCAN-Geographic…")
community_map_hdbscan_geo = run_hdbscan_geo(G, min_cluster_size=10)
print(f"  {len(set(community_map_hdbscan_geo.values()))} clusters")
plot_community_geo(
    G, community_map_hdbscan_geo,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="HDBSCAN-Geographic",
)

print("BigCLAM…")
F_bigclam, community_map_bigclam = run_bigclam(
    G, k=k_louvain, n_iter=100, lr=0.005, seed=42,
)
print(f"  {len(set(community_map_bigclam.values()))} non-empty communities")
plot_community_geo(
    G, community_map_bigclam,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=20, method_name="BigCLAM",
)

partitions_ext = {
    **partitions,
    "HDBSCAN-Spec": community_map_hdbscan_spec,
    "HDBSCAN-Geo":  community_map_hdbscan_geo,
    "BigCLAM":      community_map_bigclam,
}
nmi_ext = compute_nmi_matrix(partitions_ext)
display(nmi_ext.round(3))
plot_nmi_heatmap(nmi_ext, title="TL Italy — all methods")

scores_df = compare_partitions(G, partitions_ext, cell_size_km=10.0)
display(scores_df.round(4))
plot_partition_scores(scores_df, title="TL Italy")

## Directed Community Detection

The NVG is undirected, but time provides a natural orientation: edge $i - j$
(with $i < j$) is directed $i \to j$ (earlier event influences later).  The
directed NVG encodes a causal ordering of visibility: which past events can
"see" each future event determines who influences whom in the magnitude sequence.

Directed modularity $Q_d$ (Leicht & Newman 2008) groups events that
preferentially *send and receive* visibility flow within the same community.
Physically: if event $i$ is visible to event $j$ (forward in time) but $j$
is not visible back to $i$ (which it cannot be), the community structure is
shaped by the asymmetric amplitude ramp — foreshock episodes (rising magnitude
trend) look different from aftershock episodes (decaying trend) in the
directed graph.

**NMI vs undirected Louvain:** low NMI implies that the time direction
fundamentally reorganises which events cluster together — foreshock ramps vs
aftershock tails yield different communities when flow direction is respected.
High NMI implies the amplitude structure is temporally symmetric (no preferred
direction of visibility).

**Expected output:** directed communities ≈ similar count as undirected
(± 30%).  NMI ≈ 0.3–0.6.  If much lower, the Italian catalog shows a strong
asymmetry between foreshock and aftershock visibility — seismologically
interesting for the Amatrice 2016 sequence which had a pronounced foreshock ramp.

In [ ]:
print("Building directed NVG (time-arrow orientation: i→j for i<j)…")
G_dir = nx.DiGraph()
G_dir.add_nodes_from(G.nodes(data=True))
G_dir.add_edges_from(
    (u, v) if u < v else (v, u)
    for u, v in G.edges()
)
print(f"Directed NVG: {G_dir.number_of_nodes():,} nodes, {G_dir.number_of_edges():,} edges")

print("Running directed Louvain (Leiden) on directed NVG…")
community_map_directed = run_directed_louvain(G_dir, seed=42)
k_directed = len(set(community_map_directed.values()))
print(f"  {k_directed} directed communities  (vs {k_louvain} undirected)")

plot_community_geo(
    G_dir, community_map_directed,
    title="TL Italy",
    center_lat=41.9, center_lon=12.5, zoom=0, height=MAP_HEIGHT, width=MAP_WIDTH,
    bounds=_IT_BOUNDS,
    min_community_size=50, method_name="Directed Louvain (Leiden)",
)

partitions_full = {**partitions, "Directed Louvain": community_map_directed}
nmi_full = compute_nmi_matrix(partitions_full)
display(nmi_full.round(3))
plot_nmi_heatmap(nmi_full, title="TL Italy — directed vs undirected")

## Community Markov Flow

The Louvain partition coarse-grains the directed NVG into a $K \times K$
flow matrix $F_{ij} = \sum_{u \in C_i,\, v \in C_j} w_{uv}$.
Row-normalisation gives the Markov transition matrix $T_{ij}$: the probability
that a visibility path originating in community $i$ moves to community $j$
in the next step.

On the directed NVG, $T_{ij}$ has a **temporal reading**: it is the fraction of
time that the magnitude sequence "crosses a community boundary" moving from
seismic episode $i$ to episode $j$.  High off-diagonal $T_{ij}$ identifies
pairs of episodes frequently adjacent in the time series — possibly fore- and
aftershock sequences of the same mainshock visible to each other across the
episode boundary.

High self-retention $T_{ii} \to 1$ means the magnitude sequence rarely crosses
into other episodes; low $T_{ii}$ means community $i$ is temporally interspersed
with others (no clear amplitude barrier isolating it).

*Reference:* Rosvall, M., & Bergstrom, C. T. (2008). *PNAS*, 105(4), 1118–1123.

**Expected output:** $K$ = Louvain community count; mean self-retention
$T_{ii} \approx 0.6$–$0.9$; mean outflow entropy ≈ 1–3 bits.  The stationary
distribution $\pi$ identifies the community absorbing the most long-run
visibility flow — typically the episode with the densest aftershock cluster.

In [ ]:
from src.community_flow import (
    build_community_flow_matrix,
    compute_markov_chain,
    compute_stationary_distribution,
    community_flow_stats,
    plot_flow_heatmap,
    plot_flow_entropy,
    plot_stationary_distribution,
)

print("Building community flow matrix (Louvain partition, directed NVG)…")
flow_count_df = build_community_flow_matrix(G_dir, community_map)
T_markov  = compute_markov_chain(flow_count_df)
stat_dist = compute_stationary_distribution(T_markov)
df_flow   = community_flow_stats(T_markov, flow_count_df, stat_dist)

K_comm = T_markov.shape[0]
print(f"Markov chain: {K_comm} community states")
print(f"Mean self-retention:  {df_flow['self_retention'].mean():.3f}")
print(f"Mean outflow entropy: {df_flow['entropy'].mean():.3f} bits "
      f"(max = log₂({K_comm}) = {np.log2(K_comm):.3f})")
print(f"Dominant attractor:   Community {df_flow.iloc[0]['community']} "
      f"(π = {df_flow.iloc[0]['stationary']:.4f})")
display(df_flow)

plot_flow_heatmap(T_markov, flow_count_df, title="TL Italy")
plot_flow_entropy(df_flow, K=K_comm, title="TL Italy")
plot_stationary_distribution(df_flow, title="TL Italy")

out_path = RESULTS_DIR / "data" / "italy_tl_community_flow.csv"
df_flow.to_csv(out_path, index=False)
print(f"Community flow stats saved → {out_path}")

## Assortativity

Newman (2003) assortativity on the NVG tests whether visibility connections
preferentially link events of similar properties.

**Magnitude assortativity** $r > 0$ (assortative) would indicate that large
events are preferentially visible to other large events — consistent with
mainshock–mainshock visibility across quiescent inter-sequence periods.  In the
NVG, two large events separated by many smaller aftershocks can see each other
across the entire aftershock window, naturally creating magnitude-assortative links.

**Degree assortativity** $r < 0$ (disassortative) confirms hub-and-spoke
structure: high-degree mainshock events connect to many low-degree (degree-2 or
3) aftershock nodes buried between larger events.

**Depth assortativity** $r \approx 0$: visibility is determined by magnitude,
not depth, so depth correlation should be near zero for the NVG.

*Reference:* Newman, M. E. J. (2003). Mixing patterns in networks.
*Physical Review E*, 67(2), 026126.

**Expected output:** magnitude assortativity $r \approx +0.05$–$+0.20$
(mildly assortative — mainshocks see each other); degree assortativity
$r \approx -0.05$–$-0.15$ (disassortative hub-spoke); depth $r \approx 0$.
Three scatter panels with 2D histograms and regression lines.

In [ ]:
print("Computing assortativity (node attributes pre-attached at construction)…")
df_assort = compute_assortativity(G)
display(df_assort)
plot_assortativity(G, title="TL Italy")

print("Average nearest-neighbour degree k_nn(k):")
plot_knn(G, title="TL Italy", gamma=gamma_tl)

print("Rich-club coefficient:")
plot_rich_club(G, title="TL Italy")

## Export Results

Three output files are written:

1. **CSV** (`italy_tl_network_metrics.csv`) — one row per event with all
centrality values, coordinates, depth, and Louvain community assignment.
2. **Pickle** (`italy_tl.pkl`) — the full NetworkX undirected Graph.
Reloads in ≈ 5 seconds for comparison notebooks.
3. **GEXF** (`italy_tl.gexf`) — Gephi-compatible XML for visual exploration.
In Gephi, run ForceAtlas2 to see if the NVG layout recovers temporal ordering.

**Expected output:** CSV ≈ 215,000 rows × 12 columns; pickle ≈ 150–300 MB;
GEXF ≈ 80–200 MB.  All three paths printed on completion.

In [ ]:
df_comm = pd.DataFrame(
    [{"cell_id": n, "community": community_map[n]} for n in community_map]
)
df_final = df_centrality.merge(
    df_comm[["cell_id", "community"]], on="cell_id", how="left"
)
out_path = RESULTS_DIR / "data" / "italy_tl_network_metrics.csv"
df_final.to_csv(out_path, index=False)
print(f"Results saved → {out_path}  ({len(df_final):,} rows)")

pkl_path = RESULTS_DIR / "cache" / "italy_tl.pkl"
with open(pkl_path, "wb") as f:
    pickle.dump(G, f)
print(f"Network cached → {pkl_path}")

gexf_path = RESULTS_DIR / "gephi" / "italy_tl.gexf"
nx.write_gexf(G, gexf_path)
print(f"Gephi export → {gexf_path}")